# Grafos — Lista e Matriz de Adjacência
#jefferson - mateus -  Richard

**EDB II — UFRN — Grupo D**

Richard Matheus Bezerra Ataliba · Mateus Ferreira Torres Galvão · Jefferson Deylane Costa dos Santos

Notebook pronto para o Google Colab. Usa só a biblioteca padrão do Python: não há `pip install`. No Colab, use *Ambiente de execução → Executar tudo*. As células precisam rodar de cima para baixo.

**Grafo** é um conjunto de **vértices** (pontos) conectados por **arestas** (ligações).

O exemplo da aula tem quatro vértices, **A, B, C e D**, e as arestas **(A, B), (A, C), (B, C), (B, D), (C, D)**. Não existe a aresta (A, D).

```text
A ----- B
| \     |
|  \    |
C ----- D
```

- **Lista de adjacência:** cada vértice guarda os vizinhos com quem tem ligação.
- **Matriz de adjacência:** uma tabela com **1** quando existe ligação e **0** quando não existe.

As classes `GrafoLista` e `GrafoMatriz` cobrem grafo direcionado ou não, com peso ou sem peso, e as operações de incluir e remover vértice, incluir e remover aresta, consultar aresta, listar vizinhos, calcular grau (entrada e saída no direcionado) e imprimir o grafo.


## O que é um grafo?

Um grafo guarda entidades e as relações entre elas.

- **Vértices:** as entidades. Uma pessoa, uma cidade, um cruzamento.
- **Arestas:** as relações ou os caminhos entre entidades.
- **Não direcionado:** a ligação vale nos dois sentidos. Amizade é o caso típico: se Ana é amiga de Bruno, Bruno é amigo de Ana.
- **Direcionado:** a aresta tem um sentido. Seguir alguém numa rede social não obriga o caminho de volta.
- **Sem pesos:** toda aresta existe ou não existe. O custo é o mesmo para todas.
- **Com pesos (ponderado):** cada aresta carrega um número. Distância, tempo ou custo.

Neste notebook o grafo **sem pesos** guarda 1 na aresta. O grafo **com pesos** guarda o número informado. O peso 0 é um custo válido e é diferente de “não existe aresta”. Laços (aresta de um vértice para ele mesmo) ficam de fora, para o grau ficar fácil de ler.


## O exemplo dos slides, sem classe

Antes das classes, o mesmo grafo aparece como dicionário e como tabela. É o desenho da aula:

- lista: A → B, C · B → A, C, D · C → A, B, D · D → B, C
- matriz simétrica, com 0 na casa (A, D)

A célula abaixo repete o roteiro do exemplo inicial do grupo: imprimir as duas representações e conferir se elas concordam.


In [1]:
r"""
GRAFOS: LISTA E MATRIZ DE ADJACÊNCIA
Exemplo da aula de Estrutura de Dados II (UFRN), Grupo D.

    A ----- B
    | \     |
    |  \    |
    C ----- D

Arestas: (A, B), (A, C), (B, C), (B, D), (C, D).
Não existe aresta entre A e D.
"""

# 1) LISTA DE ADJACÊNCIA
# Para cada vértice, guardamos uma lista dos seus vizinhos.
lista = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B", "D"],
    "D": ["B", "C"],
}

# 2) MATRIZ DE ADJACÊNCIA
# A linha e a coluna representam os vértices na ordem abaixo.
# 1 = existe ligação; 0 = não existe ligação.
vertices = ["A", "B", "C", "D"]
matriz = [
    [0, 1, 1, 0],  # A ligado a B e C
    [1, 0, 1, 1],  # B ligado a A, C e D
    [1, 1, 0, 1],  # C ligado a A, B e D
    [0, 1, 1, 0],  # D ligado a B e C
]


def ligados_na_lista(origem, destino):
    """Verifica se há aresta usando a lista."""
    return destino in lista[origem]


def ligados_na_matriz(origem, destino):
    """Verifica se há aresta usando a matriz."""
    i = vertices.index(origem)
    j = vertices.index(destino)
    return matriz[i][j] == 1


print("=== LISTA DE ADJACÊNCIA ===")
for vertice, vizinhos in lista.items():
    print(f"{vertice}: {vizinhos}")

print("\n=== MATRIZ DE ADJACÊNCIA ===")
print("   ", "  ".join(vertices))
for vertice, linha in zip(vertices, matriz):
    print(vertice, " ", "  ".join(map(str, linha)))

print("\n=== TESTE SIMPLES ===")
for origem, destino in [("A", "B"), ("A", "D")]:
    resultado_lista = ligados_na_lista(origem, destino)
    resultado_matriz = ligados_na_matriz(origem, destino)
    print(
        f"{origem} está ligado a {destino}? "
        f"Lista: {resultado_lista} | Matriz: {resultado_matriz}"
    )
    assert resultado_lista == resultado_matriz, "As representações discordam!"

assert ligados_na_lista("A", "B") is True
assert ligados_na_matriz("A", "D") is False
assert ligados_na_lista("B", "C") is True
assert matriz[vertices.index("B")][vertices.index("C")] == 1
print("\nTodos os testes passaram!")

# RESUMO:
# Lista: guarda só os vizinhos; costuma economizar espaço quando há poucas arestas.
# Matriz: usa uma tabela N x N; facilita consultar se dois vértices estão ligados.


=== LISTA DE ADJACÊNCIA ===
A: ['B', 'C']
B: ['A', 'C', 'D']
C: ['A', 'B', 'D']
D: ['B', 'C']

=== MATRIZ DE ADJACÊNCIA ===
    A  B  C  D
A   0  1  1  0
B   1  0  1  1
C   1  1  0  1
D   0  1  1  0

=== TESTE SIMPLES ===
A está ligado a B? Lista: True | Matriz: True
A está ligado a D? Lista: False | Matriz: False

Todos os testes passaram!


## Das estruturas soltas para duas classes

O dicionário e a tabela mostram o grafo, mas não sabem incluir um vértice, apagar uma aresta, guardar um sentido ou um peso. Quem chama o código precisa lembrar de atualizar os dois lados de uma aresta não direcionada e de não usar o 0 como peso.

`GrafoLista` e `GrafoMatriz` encapsulam isso. As duas têm a mesma ideia de uso:

| Método | Efeito |
|---|---|
| `adicionar_vertice(v)` | inclui `v`; se já existe, não muda nada |
| `remover_vertice(v)` | remove `v` e as arestas que o tocam |
| `adicionar_aresta(u, v, peso=1)` | liga `u` a `v`; sem pesos, o peso tem de ser 1 |
| `remover_aresta(u, v)` | apaga essa ligação |
| `existe_aresta(u, v)` | `True` se a ligação existe |
| `peso(u, v)` | devolve o peso |
| `vizinhos(v)` | vizinhos; no direcionado, só as saídas |
| `predecessores(v)` | quem chega em `v` |
| `grau(v)` | vizinhos, ou entrada + saída se for direcionado |
| `grau_entrada(v)`, `grau_saida(v)` | no não direcionado, os dois valem o grau |
| `imprimir()` | escreve o grafo |
| `para_matriz()` / `para_lista()` | converte para a outra representação |

Os dois vértices precisam existir antes da aresta. Aresta repetida e laço geram `ValueError`. Remover algo que não existe gera `KeyError`. `existe_aresta` devolve `False` quando o vértice não está no grafo, porque a pergunta é “essa ligação existe?”.


## Lista de adjacência

Cada vértice aponta para a lista dos seus vizinhos. Se o grafo tem poucas arestas (esparso), a maior parte das casas de uma matriz seria zero: a lista simplesmente não guarda esses zeros.

Nesta implementação a lista de cada vértice é uma lista de pares `(vizinho, peso)`. Verificar se uma aresta existe custa **O(grau)** da origem. Acrescentar um vértice custa **O(1)**. O espaço fica **O(V + E)**: um registro por vértice e um por extremidade de aresta.

No grafo direcionado também há a lista de quem chega. Ela não muda a ordem de grandeza e faz o grau de entrada custar **O(1)**. Sem essa lista reversa, o grau de entrada varreria o grafo inteiro, **O(V + E)**.


In [2]:
from __future__ import annotations


class GrafoLista:
    """Lista de adjacência.

    Cada vértice guarda uma lista de pares (vizinho, peso), na ordem
    em que as arestas foram inseridas. O nome do vértice fica num
    dicionário, então achar o vértice é O(1) no caso médio; achar uma
    aresta percorre só os vizinhos da origem, O(grau).

    Grafo não direcionado: a aresta {u, v} é armazenada nas duas pontas.
    Grafo direcionado: além da lista de saída, guardamos a lista de
    predecessores. O espaço continua O(V + E) e o grau de entrada fica O(1).

    Laços não são aceitos. Inserir a mesma aresta duas vezes gera erro.
    Em grafo sem pesos o único peso válido é 1.
    """

    def __init__(self, direcionado: bool = False, ponderado: bool = False) -> None:
        self.direcionado = bool(direcionado)
        self.ponderado = bool(ponderado)
        self._adj: dict[object, list[tuple[object, float]]] = {}
        self._entrada: dict[object, list[tuple[object, float]]] = {}

    def vertices(self) -> list:
        """Vértices na ordem em que foram inseridos."""
        return list(self._adj)

    def adicionar_vertice(self, vertice) -> None:
        """Inclui um vértice isolado. Se ele já existe, não faz nada."""
        if vertice in self._adj:
            return
        self._adj[vertice] = []
        if self.direcionado:
            self._entrada[vertice] = []

    def remover_vertice(self, vertice) -> None:
        """Remove o vértice e toda aresta que o toca."""
        self._exigir(vertice)
        if self.direcionado:
            for destino, _peso in self._adj[vertice]:
                self._entrada[destino] = [
                    (origem, peso)
                    for origem, peso in self._entrada[destino]
                    if origem != vertice
                ]
            for origem, _peso in self._entrada[vertice]:
                self._adj[origem] = [
                    (destino, peso)
                    for destino, peso in self._adj[origem]
                    if destino != vertice
                ]
            del self._entrada[vertice]
        else:
            for vizinho, _peso in self._adj[vertice]:
                self._adj[vizinho] = [
                    (outro, peso)
                    for outro, peso in self._adj[vizinho]
                    if outro != vertice
                ]
        del self._adj[vertice]

    def adicionar_aresta(self, origem, destino, peso: float = 1) -> None:
        """Cria a aresta origem -> destino. No grafo não direcionado, vale nos dois sentidos."""
        self._exigir(origem)
        self._exigir(destino)
        if origem == destino:
            raise ValueError(
                "laço não é permitido: origem e destino são o mesmo vértice"
            )
        peso = self._normalizar_peso(peso)
        if self._buscar(origem, destino) is not None:
            raise ValueError(f"aresta já existe: {origem!r} -> {destino!r}")
        self._ligar(origem, destino, peso)
        if not self.direcionado:
            self._ligar(destino, origem, peso)

    def remover_aresta(self, origem, destino) -> None:
        """Apaga a aresta. No grafo não direcionado, apaga os dois sentidos."""
        self._exigir(origem)
        self._exigir(destino)
        if self._buscar(origem, destino) is None:
            raise KeyError(f"aresta inexistente: {origem!r} -> {destino!r}")
        self._desligar(origem, destino)
        if not self.direcionado:
            self._desligar(destino, origem)

    def existe_aresta(self, origem, destino) -> bool:
        """True se a aresta existe. Vértice ausente conta como aresta ausente."""
        if origem not in self._adj or destino not in self._adj:
            return False
        return self._buscar(origem, destino) is not None

    def peso(self, origem, destino) -> float:
        """Peso da aresta. Erro se o vértice ou a aresta não existem."""
        self._exigir(origem)
        self._exigir(destino)
        encontrado = self._buscar(origem, destino)
        if encontrado is None:
            raise KeyError(f"aresta inexistente: {origem!r} -> {destino!r}")
        return encontrado

    def vizinhos(self, vertice) -> list:
        """Adjacentes. No grafo direcionado, apenas quem sai deste vértice."""
        self._exigir(vertice)
        return [destino for destino, _peso in self._adj[vertice]]

    def predecessores(self, vertice) -> list:
        """Quem aponta para o vértice. No grafo não direcionado, são os vizinhos."""
        self._exigir(vertice)
        if not self.direcionado:
            return self.vizinhos(vertice)
        return [origem for origem, _peso in self._entrada[vertice]]

    def grau(self, vertice) -> int:
        """Não direcionado: número de vizinhos. Direcionado: entrada + saída."""
        if self.direcionado:
            return self.grau_entrada(vertice) + self.grau_saida(vertice)
        return self.grau_saida(vertice)

    def grau_saida(self, vertice) -> int:
        """Quantas arestas saem do vértice. No não direcionado, é o grau."""
        self._exigir(vertice)
        return len(self._adj[vertice])

    def grau_entrada(self, vertice) -> int:
        """Quantas arestas chegam ao vértice. No não direcionado, é o grau."""
        self._exigir(vertice)
        if not self.direcionado:
            return len(self._adj[vertice])
        return len(self._entrada[vertice])

    def arestas(self) -> list[tuple]:
        """Lista (origem, destino, peso). Cada aresta não direcionada aparece uma vez."""
        vistas: set[tuple] = set()
        resultado = []
        for origem, ligacoes in self._adj.items():
            for destino, peso in ligacoes:
                if self.direcionado:
                    chave = (origem, destino)
                    if chave in vistas:
                        continue
                    vistas.add(chave)
                else:
                    ida = (origem, destino)
                    volta = (destino, origem)
                    if ida in vistas or volta in vistas:
                        continue
                    vistas.add(ida)
                resultado.append((origem, destino, peso))
        return resultado

    def quantidade_arestas(self) -> int:
        return len(self.arestas())

    def espaco_estrutural(self) -> int:
        """Pares (vizinho, peso) guardados.

        Não direcionado: cada aresta conta nas duas pontas.
        Direcionado: conta a lista de saída e a de predecessores.
        A ordem continua O(V + E); o V dos vértices não entra neste número.
        """
        total = sum(len(ligacoes) for ligacoes in self._adj.values())
        if self.direcionado:
            total += sum(len(ligacoes) for ligacoes in self._entrada.values())
        return total

    def para_matriz(self) -> GrafoMatriz:
        """Copia este grafo para uma matriz de adjacência."""
        outro = GrafoMatriz(direcionado=self.direcionado, ponderado=self.ponderado)
        for vertice in self.vertices():
            outro.adicionar_vertice(vertice)
        for origem, destino, peso in self.arestas():
            outro.adicionar_aresta(origem, destino, peso)
        return outro

    def imprimir(self) -> None:
        """Mostra o grafo na saída padrão."""
        print(self)

    def __str__(self) -> str:
        linhas = [self._descricao()]
        if not self._adj:
            linhas.append("(grafo vazio)")
            return "\n".join(linhas)
        for vertice, ligacoes in self._adj.items():
            if not ligacoes:
                linhas.append(f"{vertice} → (sem vizinhos)")
                continue
            if self.ponderado:
                itens = ", ".join(
                    f"{destino}({_formatar_numero(peso)})" for destino, peso in ligacoes
                )
            else:
                itens = ", ".join(str(destino) for destino, _peso in ligacoes)
            linhas.append(f"{vertice} → {itens}")
        return "\n".join(linhas)

    def __repr__(self) -> str:
        return (
            f"GrafoLista(direcionado={self.direcionado}, "
            f"ponderado={self.ponderado}, vertices={len(self._adj)})"
        )

    def _descricao(self) -> str:
        return _descricao_grafo("Lista de adjacência", self.direcionado, self.ponderado)

    def _exigir(self, vertice) -> None:
        if vertice not in self._adj:
            raise KeyError(
                f"vértice inexistente: {vertice!r}. Use adicionar_vertice primeiro."
            )

    def _normalizar_peso(self, peso):
        return _normalizar_peso(self.ponderado, peso)

    def _buscar(self, origem, destino):
        for vizinho, peso in self._adj[origem]:
            if vizinho == destino:
                return peso
        return None

    def _ligar(self, origem, destino, peso) -> None:
        self._adj[origem].append((destino, peso))
        if self.direcionado:
            self._entrada[destino].append((origem, peso))

    def _desligar(self, origem, destino) -> None:
        self._adj[origem] = [
            (vizinho, peso) for vizinho, peso in self._adj[origem] if vizinho != destino
        ]
        if self.direcionado:
            self._entrada[destino] = [
                (vizinho, peso)
                for vizinho, peso in self._entrada[destino]
                if vizinho != origem
            ]


def _descricao_grafo(nome: str, direcionado: bool, ponderado: bool) -> str:
    sentido = "direcionado" if direcionado else "não direcionado"
    pesos = "com pesos" if ponderado else "sem pesos"
    return f"{nome} | {sentido} | {pesos}"


def _normalizar_peso(ponderado: bool, peso):
    if isinstance(peso, bool) or not isinstance(peso, (int, float)):
        raise TypeError("o peso precisa ser um número")
    if not ponderado:
        if peso != 1:
            raise ValueError("grafo sem pesos só aceita peso 1")
        return 1
    if isinstance(peso, float):
        return float(peso)
    return int(peso)


def _formatar_numero(valor) -> str:
    if isinstance(valor, float) and valor.is_integer():
        return str(int(valor))
    return str(valor)


## Matriz de adjacência

A matriz é uma tabela V × V. A casa da linha de `u` com a coluna de `v` responde, em **O(1)**, se a aresta existe e qual é o peso. O preço é a memória: as casas existem mesmo quando a aresta não existe, então o espaço é **O(V²)**. Listar os vizinhos olha a linha toda e custa **O(V)**, até quando o vértice tem um único vizinho.

Incluir um vértice acrescenta uma casa em cada linha já existente e cria uma linha nova: **O(V)**. Remover um vértice apaga uma linha e uma coluna. Apagar a coluna de cada linha custa **O(V²)**.

Sem pesos, a tabela usa 0 e 1, como nos slides. Com pesos, a casa vazia fica `None` e aparece como `-` na impressão, para o peso 0 continuar visível. No grafo não direcionado a matriz é simétrica: o que vale de A para B vale de B para A.


In [3]:
class GrafoMatriz:
    """Matriz de adjacência.

    A linha i e a coluna j representam vértices. A casa M[i][j] guarda
    o peso da aresta i -> j. Sem pesos, 1 significa aresta e 0 significa
    ausência (como nos slides). Com pesos, None significa ausência, para
    não confundir com o peso 0.

    Consultar uma casa é O(1). Achar o índice a partir do nome também é
    O(1) no caso médio, porque os nomes ficam num dicionário. Listar
    vizinhos olha a linha inteira: O(V). O espaço é sempre O(V²), com
    ou sem arestas.

    Grafo não direcionado: M[i][j] e M[j][i] recebem o mesmo valor,
    então a matriz fica simétrica.
    """

    def __init__(self, direcionado: bool = False, ponderado: bool = False) -> None:
        self.direcionado = bool(direcionado)
        self.ponderado = bool(ponderado)
        self._vertices: list = []
        self._indice: dict = {}
        self._matriz: list[list] = []

    def vertices(self) -> list:
        """Vértices na ordem das linhas e das colunas."""
        return list(self._vertices)

    def adicionar_vertice(self, vertice) -> None:
        """Acrescenta uma linha e uma coluna. Se o vértice já existe, não faz nada."""
        if vertice in self._indice:
            return
        self._indice[vertice] = len(self._vertices)
        self._vertices.append(vertice)
        vazio = self._vazio()
        for linha in self._matriz:
            linha.append(vazio)
        self._matriz.append([vazio for _ in range(len(self._vertices))])

    def remover_vertice(self, vertice) -> None:
        """Apaga a linha e a coluna do vértice e reindexa os demais."""
        self._exigir(vertice)
        indice = self._indice[vertice]
        del self._vertices[indice]
        del self._matriz[indice]
        for linha in self._matriz:
            del linha[indice]
        self._indice = {nome: posicao for posicao, nome in enumerate(self._vertices)}

    def adicionar_aresta(self, origem, destino, peso: float = 1) -> None:
        """Grava M[origem][destino]. No grafo não direcionado, grava também o simétrico."""
        self._exigir(origem)
        self._exigir(destino)
        if origem == destino:
            raise ValueError(
                "laço não é permitido: origem e destino são o mesmo vértice"
            )
        peso = _normalizar_peso(self.ponderado, peso)
        linha = self._indice[origem]
        coluna = self._indice[destino]
        if self._matriz[linha][coluna] != self._vazio():
            raise ValueError(f"aresta já existe: {origem!r} -> {destino!r}")
        self._matriz[linha][coluna] = peso
        if not self.direcionado:
            self._matriz[coluna][linha] = peso

    def remover_aresta(self, origem, destino) -> None:
        """Zera a casa da aresta. No grafo não direcionado, zera as duas casas."""
        self._exigir(origem)
        self._exigir(destino)
        linha = self._indice[origem]
        coluna = self._indice[destino]
        if self._matriz[linha][coluna] == self._vazio():
            raise KeyError(f"aresta inexistente: {origem!r} -> {destino!r}")
        vazio = self._vazio()
        self._matriz[linha][coluna] = vazio
        if not self.direcionado:
            self._matriz[coluna][linha] = vazio

    def existe_aresta(self, origem, destino) -> bool:
        """True se M[origem][destino] está ocupada. Vértice ausente conta como ausência."""
        if origem not in self._indice or destino not in self._indice:
            return False
        linha = self._indice[origem]
        coluna = self._indice[destino]
        return self._matriz[linha][coluna] != self._vazio()

    def peso(self, origem, destino) -> float:
        """Valor guardado na casa. Erro se o vértice ou a aresta não existem."""
        self._exigir(origem)
        self._exigir(destino)
        linha = self._indice[origem]
        coluna = self._indice[destino]
        valor = self._matriz[linha][coluna]
        if valor == self._vazio():
            raise KeyError(f"aresta inexistente: {origem!r} -> {destino!r}")
        return valor

    def vizinhos(self, vertice) -> list:
        """Vértices da linha com aresta. No direcionado, são as saídas."""
        self._exigir(vertice)
        linha = self._indice[vertice]
        vazio = self._vazio()
        return [
            self._vertices[coluna]
            for coluna, valor in enumerate(self._matriz[linha])
            if valor != vazio
        ]

    def predecessores(self, vertice) -> list:
        """Vértices da coluna com aresta. No não direcionado, são os vizinhos."""
        self._exigir(vertice)
        if not self.direcionado:
            return self.vizinhos(vertice)
        coluna = self._indice[vertice]
        vazio = self._vazio()
        return [
            self._vertices[linha]
            for linha, valores in enumerate(self._matriz)
            if valores[coluna] != vazio
        ]

    def grau(self, vertice) -> int:
        """Não direcionado: vizinhos da linha. Direcionado: entrada + saída."""
        if self.direcionado:
            return self.grau_entrada(vertice) + self.grau_saida(vertice)
        return self.grau_saida(vertice)

    def grau_saida(self, vertice) -> int:
        """Casas ocupadas na linha. Custa O(V)."""
        self._exigir(vertice)
        linha = self._indice[vertice]
        vazio = self._vazio()
        return sum(valor != vazio for valor in self._matriz[linha])

    def grau_entrada(self, vertice) -> int:
        """Casas ocupadas na coluna. Custa O(V). No não direcionado, é o grau."""
        self._exigir(vertice)
        if not self.direcionado:
            return self.grau_saida(vertice)
        coluna = self._indice[vertice]
        vazio = self._vazio()
        return sum(linha[coluna] != vazio for linha in self._matriz)

    def arestas(self) -> list[tuple]:
        """Lista (origem, destino, peso). Cada aresta não direcionada aparece uma vez."""
        resultado = []
        quantidade = len(self._vertices)
        vazio = self._vazio()
        for linha in range(quantidade):
            inicio = 0 if self.direcionado else linha + 1
            for coluna in range(inicio, quantidade):
                valor = self._matriz[linha][coluna]
                if valor != vazio:
                    resultado.append(
                        (self._vertices[linha], self._vertices[coluna], valor)
                    )
        return resultado

    def quantidade_arestas(self) -> int:
        return len(self.arestas())

    def espaco_estrutural(self) -> int:
        """Número de casas da tabela, ocupadas ou não: sempre V²."""
        return len(self._vertices) ** 2

    def como_tabela(self) -> list[list]:
        """Cópia das casas. Sem pesos: 0 ou 1. Com pesos: None ou o peso."""
        return [list(linha) for linha in self._matriz]

    def para_lista(self) -> GrafoLista:
        """Copia esta matriz para uma lista de adjacência."""
        outro = GrafoLista(direcionado=self.direcionado, ponderado=self.ponderado)
        for vertice in self.vertices():
            outro.adicionar_vertice(vertice)
        for origem, destino, peso in self.arestas():
            outro.adicionar_aresta(origem, destino, peso)
        return outro

    def imprimir(self) -> None:
        """Mostra o grafo na saída padrão."""
        print(self)

    def __str__(self) -> str:
        linhas = [self._descricao()]
        if not self._vertices:
            linhas.append("(grafo vazio)")
            return "\n".join(linhas)
        linhas.append(self._formatar_tabela())
        return "\n".join(linhas)

    def __repr__(self) -> str:
        return (
            f"GrafoMatriz(direcionado={self.direcionado}, "
            f"ponderado={self.ponderado}, vertices={len(self._vertices)})"
        )

    def _descricao(self) -> str:
        return _descricao_grafo("Matriz de adjacência", self.direcionado, self.ponderado)

    def _formatar_tabela(self) -> str:
        textos = [[self._texto_celula(valor) for valor in linha] for linha in self._matriz]
        nomes = [str(vertice) for vertice in self._vertices]
        largura = 1
        for nome in nomes:
            largura = max(largura, len(nome))
        for linha in textos:
            for texto in linha:
                largura = max(largura, len(texto))
        separador = "  "
        cabecalho = " " * largura + separador + separador.join(
            nome.rjust(largura) for nome in nomes
        )
        corpo = []
        for nome, linha in zip(nomes, textos):
            casas = separador.join(texto.rjust(largura) for texto in linha)
            corpo.append(nome.rjust(largura) + separador + casas)
        return "\n".join([cabecalho, *corpo])

    def _texto_celula(self, valor) -> str:
        if valor == self._vazio():
            return "-" if self.ponderado else "0"
        return _formatar_numero(valor)

    def _vazio(self):
        return None if self.ponderado else 0

    def _exigir(self, vertice) -> None:
        if vertice not in self._indice:
            raise KeyError(
                f"vértice inexistente: {vertice!r}. Use adicionar_vertice primeiro."
            )


## Complexidade: tempo e espaço

`V` é o número de vértices, `E` é o número de arestas e `grau` é a quantidade de vizinhos do vértice consultado. Os tempos são os desta implementação.

| Operação | Lista de adjacência | Matriz de adjacência |
|---|---|---|
| Espaço | O(V + E) | O(V²) |
| Adicionar vértice | O(1) | O(V) |
| Remover vértice | O(V + E) | O(V²) |
| Adicionar aresta | O(grau) | O(1) |
| Remover aresta | O(grau) | O(1) |
| Existe aresta? | O(grau) | O(1) |
| Consultar peso | O(grau) | O(1) |
| Listar vizinhos | O(grau) | O(V) |
| Grau / grau de saída | O(1) | O(V) |
| Grau de entrada | O(1), com a lista de predecessores | O(V) |

Como ler a tabela: **O(1)** não cresce com o grafo; **O(grau)** cresce com os vizinhos daquele vértice; **O(V)** cresce com todos os vértices; **O(V²)** cresce com o quadrado dos vértices.

O nome do vértice vira posição da matriz por um dicionário, O(1) no caso médio. A casa `M[i][j]` em si é um acesso direto. Na lista, o dicionário também acha o vértice em O(1) no caso médio; o que depende do grau é a varredura dos vizinhos.

A lista é a escolha natural quando `E` é bem menor que `V²` (rede social, malha de ruas). A matriz compensa quando o grafo é denso ou quando o programa pergunta o tempo todo se dois vértices específicos estão ligados e `V` cabe na memória.


## O grafo da aula nas duas classes

A célula constrói o grafo não direcionado e sem pesos dos slides, primeiro na lista e depois na matriz, com as mesmas arestas. A lista deve imprimir os vizinhos da aula e a matriz deve repetir a tabela 0/1, simétrica, com zero entre A e D.


In [4]:
VERTICES_AULA = ["A", "B", "C", "D"]
ARESTAS_AULA = [("A", "B"), ("A", "C"), ("B", "C"), ("B", "D"), ("C", "D")]
MATRIZ_AULA = [
    [0, 1, 1, 0],
    [1, 0, 1, 1],
    [1, 1, 0, 1],
    [0, 1, 1, 0],
]


def montar_grafo(classe, vertices, arestas, direcionado=False, ponderado=False):
    """Cria um grafo, insere os vértices e depois as arestas, nesta ordem."""
    grafo = classe(direcionado=direcionado, ponderado=ponderado)
    for vertice in vertices:
        grafo.adicionar_vertice(vertice)
    for aresta in arestas:
        grafo.adicionar_aresta(*aresta)
    return grafo


grafo_lista = montar_grafo(GrafoLista, VERTICES_AULA, ARESTAS_AULA)
grafo_matriz = montar_grafo(GrafoMatriz, VERTICES_AULA, ARESTAS_AULA)

print(grafo_lista)
print()
print(grafo_matriz)

assert grafo_lista.vizinhos("A") == ["B", "C"]
assert grafo_lista.vizinhos("B") == ["A", "C", "D"]
assert grafo_lista.vizinhos("C") == ["A", "B", "D"]
assert grafo_lista.vizinhos("D") == ["B", "C"]
assert grafo_lista.grau("A") == 2
assert grafo_lista.grau("B") == 3
assert grafo_lista.grau("C") == 3
assert grafo_lista.grau("D") == 2
assert grafo_lista.existe_aresta("A", "B") is True
assert grafo_lista.existe_aresta("B", "A") is True
assert grafo_lista.existe_aresta("A", "D") is False
assert grafo_lista.quantidade_arestas() == 5

assert grafo_matriz.como_tabela() == MATRIZ_AULA
assert grafo_matriz.existe_aresta("B", "C") is True
assert grafo_matriz.existe_aresta("A", "D") is False
assert grafo_matriz.grau("C") == 3
assert grafo_matriz.quantidade_arestas() == 5
for i, linha in enumerate(MATRIZ_AULA):
    for j, valor in enumerate(linha):
        assert valor == MATRIZ_AULA[j][i], "a matriz do grafo não direcionado é simétrica"

print()
print("Lista e matriz representam o mesmo grafo da aula.")


Lista de adjacência | não direcionado | sem pesos
A → B, C
B → A, C, D
C → A, B, D
D → B, C

Matriz de adjacência | não direcionado | sem pesos
   A  B  C  D
A  0  1  1  0
B  1  0  1  1
C  1  1  0  1
D  0  1  1  0

Lista e matriz representam o mesmo grafo da aula.


## Grafo direcionado com pesos

O desenho abaixo é um mapa pequeno de ruas de mão única. O peso é o tempo em minutos. A seta A → B não cria a rua B → A.

```text
A --3--> B --4--> D
|        ^        ^
1        |        |
v        5        2
C -------+--------+
```

Arestas: A→B (3), A→C (1), B→D (4), C→B (5), C→D (2).

D recebe duas ruas, então o grau de entrada de D é 2. A só tem saídas: grau de entrada 0 e grau de saída 2. O grau total de um vértice direcionado é a soma dos dois.


In [5]:
vertices_rotas = ["A", "B", "C", "D"]
rotas = [
    ("A", "B", 3),
    ("A", "C", 1),
    ("B", "D", 4),
    ("C", "B", 5),
    ("C", "D", 2),
]

rotas_lista = montar_grafo(
    GrafoLista,
    vertices_rotas,
    rotas,
    direcionado=True,
    ponderado=True,
)
rotas_matriz = montar_grafo(
    GrafoMatriz,
    vertices_rotas,
    rotas,
    direcionado=True,
    ponderado=True,
)

print(rotas_lista)
print()
print(rotas_matriz)

assert rotas_lista.existe_aresta("A", "B") is True
assert rotas_lista.existe_aresta("B", "A") is False
assert rotas_lista.peso("B", "D") == 4
assert rotas_lista.grau_saida("A") == 2
assert rotas_lista.grau_entrada("A") == 0
assert rotas_lista.grau_entrada("D") == 2
assert rotas_lista.grau_saida("D") == 0
assert rotas_lista.grau("B") == rotas_lista.grau_entrada("B") + rotas_lista.grau_saida("B")
assert rotas_lista.grau_entrada("B") == 2
assert sorted(rotas_lista.predecessores("B")) == ["A", "C"]

assert rotas_matriz.como_tabela() == [
    [None, 3, 1, None],
    [None, None, None, 4],
    [None, 5, None, 2],
    [None, None, None, None],
]
assert rotas_matriz.peso("C", "D") == 2
assert rotas_matriz.existe_aresta("D", "C") is False
assert rotas_matriz.grau_entrada("D") == 2
assert rotas_matriz.grau_saida("C") == 2

print()
print("O sentido importa: A alcança B em 3 min e B não tem volta para A.")


Lista de adjacência | direcionado | com pesos
A → B(3), C(1)
B → D(4)
C → B(5), D(2)
D → (sem vizinhos)

Matriz de adjacência | direcionado | com pesos
   A  B  C  D
A  -  3  1  -
B  -  -  -  4
C  -  5  -  2
D  -  -  -  -

O sentido importa: A alcança B em 3 min e B não tem volta para A.


## Conversão lista ↔ matriz

As duas classes guardam o mesmo grafo de jeitos diferentes. `para_matriz` e `para_lista` copiam vértices, sentido, pesos e arestas.

A ordem dos vizinhos pode mudar na volta: a lista lembra a ordem de inserção e a matriz devolve os vizinhos na ordem das colunas. O conjunto de arestas e os pesos permanecem. A célula confere o grafo da aula e o mapa de ruas nos dois caminhos.


In [6]:
def assinatura(grafo):
    """Conteúdo do grafo, com vizinhos ordenados para comparar as representações."""
    vertices = grafo.vertices()
    return {
        "direcionado": grafo.direcionado,
        "ponderado": grafo.ponderado,
        "vertices": vertices,
        "saidas": {
            vertice: sorted(
                (str(vizinho), grafo.peso(vertice, vizinho))
                for vizinho in grafo.vizinhos(vertice)
            )
            for vertice in vertices
        },
        "entradas": {
            vertice: sorted(
                (str(origem), grafo.peso(origem, vertice))
                for origem in grafo.predecessores(vertice)
            )
            for vertice in vertices
        },
    }


def ida_e_volta(grafo):
    """Lista vira matriz e volta; matriz vira lista e volta."""
    if isinstance(grafo, GrafoLista):
        matriz = grafo.para_matriz()
        lista = matriz.para_lista()
    else:
        lista = grafo.para_lista()
        matriz = lista.para_matriz()
    return lista, matriz


print("Grafo da aula, lista → matriz → lista")
print(grafo_lista.para_matriz())
lista_aula, matriz_aula = ida_e_volta(grafo_lista)
assert isinstance(lista_aula, GrafoLista)
assert isinstance(matriz_aula, GrafoMatriz)
assert assinatura(lista_aula) == assinatura(grafo_lista)
assert assinatura(matriz_aula) == assinatura(grafo_matriz)
assert matriz_aula.como_tabela() == MATRIZ_AULA

print()
print("Ruas de mão única, matriz → lista → matriz")
lista_rotas, matriz_rotas = ida_e_volta(rotas_matriz)
assert assinatura(lista_rotas) == assinatura(rotas_lista)
assert assinatura(matriz_rotas) == assinatura(rotas_matriz)
assert matriz_rotas.como_tabela() == rotas_matriz.como_tabela()
assert lista_rotas.peso("A", "B") == 3
assert lista_rotas.existe_aresta("B", "A") is False

print()
print("A conversão preserva vértices, sentido, arestas e pesos.")


Grafo da aula, lista → matriz → lista
Matriz de adjacência | não direcionado | sem pesos
   A  B  C  D
A  0  1  1  0
B  1  0  1  1
C  1  1  0  1
D  0  1  1  0

Ruas de mão única, matriz → lista → matriz

A conversão preserva vértices, sentido, arestas e pesos.


## Tempo e memória: grafo esparso e grafo denso

A tabela de complexidade descreve o crescimento. Esta célula mede, no Python, dois casos do grafo não direcionado e sem pesos:

- **Esparso:** um ciclo. `V` vértices e `V` arestas. Cada vértice tem grau 2.
- **Denso:** um grafo completo. Todo par de vértices tem aresta, então `E = V(V − 1) / 2`.

Há duas medidas, de propósito:

- **Espaço estrutural:** quantos pares a lista guarda, ou quantas casas a matriz reserva (`V²`). É determinístico e segue a tabela.
- **Tempo e pico do `tracemalloc`:** o que o processo alocou ao construir o grafo e ao perguntar se as arestas existem. Entram o custo dos objetos do Python e objetos temporários. No Colab os milissegundos mudam; a comparação entre lista e matriz, no mesmo processo, é o que interessa.

No esparso, a lista guarda `2V` pontas de aresta e a matriz reserva `V²` casas. No completo, a lista guarda `V(V − 1)` pontas (cada aresta nas duas extremidades) e a matriz reserva `V²`: a diferença estrutural é só a diagonal, sem aresta. Mesmo assim a lista densa pode gastar mais memória real, porque cada vizinho é uma tupla e a matriz reaproveita os inteiros 0 e 1.


In [7]:
import gc
import time
import tracemalloc


def arestas_ciclo(quantidade):
    """Ciclo: cada vértice ligado ao próximo, e o último ligado ao primeiro."""
    return [(f"v{i}", f"v{(i + 1) % quantidade}") for i in range(quantidade)]


def arestas_completo(quantidade):
    """Todos os pares i < j, uma vez. O grafo não direcionado replica o outro sentido."""
    return [
        (f"v{i}", f"v{j}")
        for i in range(quantidade)
        for j in range(i + 1, quantidade)
    ]


def construir(classe, quantidade, arestas):
    grafo = classe(direcionado=False, ponderado=False)
    for indice in range(quantidade):
        grafo.adicionar_vertice(f"v{indice}")
    for origem, destino in arestas:
        grafo.adicionar_aresta(origem, destino)
    return grafo


def medir(funcao):
    """Tempo em segundos e pico de alocação em bytes, durante a chamada."""
    gc.collect()
    if tracemalloc.is_tracing():
        tracemalloc.stop()
    tracemalloc.start()
    inicio = time.perf_counter()
    try:
        funcao()
        return time.perf_counter() - inicio, tracemalloc.get_traced_memory()[1]
    finally:
        tracemalloc.stop()


def consultar(grafo, pares):
    for origem, destino in pares:
        grafo.existe_aresta(origem, destino)


def medir_caso(rotulo, quantidade, arestas, repeticoes_consulta):
    resultados = {}
    for nome, classe in (("lista", GrafoLista), ("matriz", GrafoMatriz)):
        tempo_construir, pico = medir(lambda classe=classe: construir(classe, quantidade, arestas))
        grafo = construir(classe, quantidade, arestas)
        tempo_consultar, _pico_consulta = medir(
            lambda grafo=grafo: [consultar(grafo, arestas) for _ in range(repeticoes_consulta)]
        )
        resultados[nome] = {
            "grafo": grafo,
            "tempo_construir": tempo_construir,
            "pico": pico,
            "tempo_consultar": tempo_consultar,
            "espaco": grafo.espaco_estrutural(),
        }
    return resultados


N_ESPARSO = 300
N_DENSO = 48
CICLO = arestas_ciclo(N_ESPARSO)
COMPLETO = arestas_completo(N_DENSO)

esparso = medir_caso("esparso", N_ESPARSO, CICLO, repeticoes_consulta=40)
denso = medir_caso("denso", N_DENSO, COMPLETO, repeticoes_consulta=8)

assert esparso["lista"]["espaco"] == 2 * N_ESPARSO
assert esparso["matriz"]["espaco"] == N_ESPARSO ** 2
assert esparso["lista"]["espaco"] < esparso["matriz"]["espaco"]
assert esparso["matriz"]["pico"] > esparso["lista"]["pico"] * 2

assert len(COMPLETO) == N_DENSO * (N_DENSO - 1) // 2
assert denso["lista"]["espaco"] == N_DENSO * (N_DENSO - 1)
assert denso["matriz"]["espaco"] == N_DENSO ** 2
assert denso["lista"]["espaco"] < denso["matriz"]["espaco"]
assert denso["lista"]["grafo"].quantidade_arestas() == len(COMPLETO)
assert denso["matriz"]["grafo"].quantidade_arestas() == len(COMPLETO)
assert esparso["lista"]["grafo"].grau("v0") == 2
assert denso["matriz"]["grafo"].grau("v0") == N_DENSO - 1


def kib(quantidade_bytes):
    return quantidade_bytes / 1024


def ms(segundos):
    return segundos * 1000


print(
    f"{'caso':<22} {'repr.':<8} {'casas':>10} "
    f"{'construir ms':>14} {'consulta ms':>12} {'pico KiB':>10}"
)
for rotulo, quantidade, dados in (
    (f"esparso ciclo V={N_ESPARSO}", N_ESPARSO, esparso),
    (f"denso completo V={N_DENSO}", N_DENSO, denso),
):
    for nome in ("lista", "matriz"):
        item = dados[nome]
        print(
            f"{rotulo:<22} {nome:<8} {item['espaco']:>10} "
            f"{ms(item['tempo_construir']):>14.2f} "
            f"{ms(item['tempo_consultar']):>12.2f} "
            f"{kib(item['pico']):>10.1f}"
        )

print()
print(
    f"Esparso: a lista guarda {esparso['lista']['espaco']} pontas de aresta; "
    f"a matriz reserva {esparso['matriz']['espaco']} casas."
)
print(
    "O pico de memória da matriz esparsa ficou maior: a tabela existe mesmo sem aresta."
)
print(
    f"Denso: a lista guarda {denso['lista']['espaco']} pontas e a matriz reserva "
    f"{denso['matriz']['espaco']} casas. A folga estrutural é a diagonal."
)
if denso["lista"]["pico"] > denso["matriz"]["pico"]:
    print(
        "Na memória real deste processo, a lista densa passou a matriz: "
        "cada vizinho é um objeto, e a matriz repete os inteiros 0 e 1."
    )
else:
    print(
        "Na memória real deste processo, a matriz densa ficou à frente da lista. "
        "Com V maior, o V² da matriz passa a dominar o overhead dos objetos da lista."
    )
if esparso["lista"]["tempo_construir"] < esparso["matriz"]["tempo_construir"]:
    print(
        "Construir o ciclo foi mais rápido na lista: ela não aloca as casas vazias da matriz."
    )
else:
    print(
        "Construir o ciclo ficou mais rápido na matriz neste processo. "
        "Com V maior, alocar V² casas passa a custar mais do que encadear as poucas arestas."
    )
if denso["matriz"]["tempo_construir"] <= denso["lista"]["tempo_construir"]:
    print(
        "Construir o grafo completo foi mais rápido na matriz: "
        "gravar uma casa não percorre a lista de vizinhos."
    )
else:
    print(
        "Construir o grafo completo ficou mais rápido na lista neste processo."
    )
print(
    "A consulta na lista percorre o grau; na matriz, acessa uma casa. "
    "A diferença de tempo de consulta cresce quando o grau é alto."
)


caso                   repr.         casas   construir ms  consulta ms   pico KiB
esparso ciclo V=300    lista           600           1.70         7.39       78.7
esparso ciclo V=300    matriz        90000           6.20         1.71      764.6
denso completo V=48    lista          2256           3.59         8.48      149.7
denso completo V=48    matriz         2304           0.92         1.29       27.2

Esparso: a lista guarda 600 pontas de aresta; a matriz reserva 90000 casas.
O pico de memória da matriz esparsa ficou maior: a tabela existe mesmo sem aresta.
Denso: a lista guarda 2256 pontas e a matriz reserva 2304 casas. A folga estrutural é a diagonal.
Na memória real deste processo, a lista densa passou a matriz: cada vizinho é um objeto, e a matriz repete os inteiros 0 e 1.
Construir o ciclo foi mais rápido na lista: ela não aloca as casas vazias da matriz.
Construir o grafo completo foi mais rápido na matriz: gravar uma casa não percorre a lista de vizinhos.
A consulta na lis

## Testes

A célula monta os grafos de novo e confere, com `assert`, o contrato das duas classes: o exemplo da aula, edição de vértices e arestas, grafo direcionado com peso (inclusive peso 0), conversão, erros esperados, vértice isolado e a impressão. Se alguma afirmação falhar, a execução para nessa linha.


In [8]:
def montar(classe, vertices, arestas, direcionado=False, ponderado=False):
    grafo = classe(direcionado=direcionado, ponderado=ponderado)
    for vertice in vertices:
        grafo.adicionar_vertice(vertice)
    for aresta in arestas:
        grafo.adicionar_aresta(*aresta)
    return grafo


def conteudo(grafo):
    vertices = grafo.vertices()
    return {
        "direcionado": grafo.direcionado,
        "ponderado": grafo.ponderado,
        "vertices": vertices,
        "saidas": {
            vertice: sorted(
                (str(vizinho), grafo.peso(vertice, vizinho))
                for vizinho in grafo.vizinhos(vertice)
            )
            for vertice in vertices
        },
        "entradas": {
            vertice: sorted(
                (str(origem), grafo.peso(origem, vertice))
                for origem in grafo.predecessores(vertice)
            )
            for vertice in vertices
        },
        "graus": {
            vertice: (
                grafo.grau(vertice),
                grafo.grau_entrada(vertice),
                grafo.grau_saida(vertice),
            )
            for vertice in vertices
        },
    }


def exige(excecao, funcao):
    try:
        funcao()
    except excecao:
        return
    raise AssertionError(f"esperava {excecao.__name__}")


vertices = ["A", "B", "C", "D"]
arestas = [("A", "B"), ("A", "C"), ("B", "C"), ("B", "D"), ("C", "D")]
tabela_aula = [
    [0, 1, 1, 0],
    [1, 0, 1, 1],
    [1, 1, 0, 1],
    [0, 1, 1, 0],
]

for classe in (GrafoLista, GrafoMatriz):
    grafo = montar(classe, vertices, arestas)
    assert grafo.direcionado is False
    assert grafo.ponderado is False
    assert grafo.vertices() == vertices
    assert grafo.quantidade_arestas() == 5
    assert grafo.existe_aresta("A", "B") is True
    assert grafo.existe_aresta("B", "A") is True
    assert grafo.existe_aresta("B", "C") is True
    assert grafo.existe_aresta("A", "D") is False
    assert grafo.existe_aresta("A", "Z") is False
    assert grafo.peso("C", "D") == 1
    assert grafo.grau("A") == 2
    assert grafo.grau("B") == 3
    assert grafo.grau_entrada("D") == 2
    assert grafo.grau_saida("D") == 2
    assert sorted(grafo.vizinhos("D")) == ["B", "C"]
    assert sorted(grafo.predecessores("D")) == sorted(grafo.vizinhos("D"))
    assert {tuple(aresta[:2]) for aresta in grafo.arestas()} == {
        ("A", "B"),
        ("A", "C"),
        ("B", "C"),
        ("B", "D"),
        ("C", "D"),
    }

lista_aula = montar(GrafoLista, vertices, arestas)
matriz_aula = montar(GrafoMatriz, vertices, arestas)
assert lista_aula.vizinhos("A") == ["B", "C"]
assert lista_aula.vizinhos("B") == ["A", "C", "D"]
assert lista_aula.vizinhos("C") == ["A", "B", "D"]
assert lista_aula.vizinhos("D") == ["B", "C"]
assert lista_aula.espaco_estrutural() == 10
assert matriz_aula.como_tabela() == tabela_aula
assert matriz_aula.espaco_estrutural() == 16
assert conteudo(lista_aula.para_matriz()) == conteudo(matriz_aula)
assert conteudo(matriz_aula.para_lista()) == conteudo(lista_aula)
assert matriz_aula.para_lista().para_matriz().como_tabela() == tabela_aula

# Remover aresta não direcionada apaga os dois sentidos.
editada = montar(GrafoLista, vertices, arestas)
editada.remover_aresta("A", "B")
assert editada.existe_aresta("A", "B") is False
assert editada.existe_aresta("B", "A") is False
assert editada.vizinhos("A") == ["C"]
assert editada.vizinhos("B") == ["C", "D"]
assert editada.grau("A") == 1
assert editada.grau("B") == 2
assert editada.quantidade_arestas() == 4
espelho = editada.para_matriz()
assert espelho.existe_aresta("A", "B") is False
assert espelho.como_tabela()[0][1] == 0
assert espelho.como_tabela()[1][0] == 0

# Remover C deixa o caminho A—B—D e a aresta que não tocava C.
sem_c = montar(GrafoMatriz, vertices, arestas)
sem_c.remover_vertice("C")
assert sem_c.vertices() == ["A", "B", "D"]
assert sem_c.como_tabela() == [
    [0, 1, 0],
    [1, 0, 1],
    [0, 1, 0],
]
assert sem_c.existe_aresta("A", "B") is True
assert sem_c.existe_aresta("B", "D") is True
assert sem_c.existe_aresta("A", "D") is False
assert sem_c.existe_aresta("B", "C") is False
sem_c_lista = montar(GrafoLista, vertices, arestas)
sem_c_lista.remover_vertice("C")
assert sem_c_lista.vertices() == ["A", "B", "D"]
assert sem_c_lista.vizinhos("A") == ["B"]
assert sem_c_lista.vizinhos("B") == ["A", "D"]
assert sem_c_lista.vizinhos("D") == ["B"]
assert conteudo(sem_c_lista) == conteudo(sem_c)

# Vértice novo nasce isolado; repetir a inclusão não duplica.
isolado = montar(GrafoLista, ["A"], [])
isolado.adicionar_vertice("A")
isolado.adicionar_vertice("E")
assert isolado.vertices() == ["A", "E"]
assert isolado.grau("E") == 0
assert isolado.vizinhos("E") == []
assert isolado.existe_aresta("A", "E") is False
isolado.adicionar_aresta("E", "A")
assert isolado.grau("E") == 1
assert isolado.vizinhos("A") == ["E"]

# Direcionado e ponderado, com peso 0 diferente de aresta ausente.
rotas = [("A", "B", 3), ("A", "C", 1), ("B", "D", 4), ("C", "B", 0), ("C", "D", 2.5)]
for classe in (GrafoLista, GrafoMatriz):
    grafo = montar(classe, vertices, rotas, direcionado=True, ponderado=True)
    assert grafo.existe_aresta("A", "B") is True
    assert grafo.existe_aresta("B", "A") is False
    assert grafo.peso("C", "B") == 0
    assert grafo.peso("C", "D") == 2.5
    assert grafo.grau_saida("A") == 2
    assert grafo.grau_entrada("A") == 0
    assert grafo.grau_entrada("B") == 2
    assert grafo.grau_saida("B") == 1
    assert grafo.grau("B") == 3
    assert grafo.grau_entrada("D") == 2
    assert sorted(grafo.predecessores("B")) == ["A", "C"]
    assert sorted(grafo.vizinhos("C")) == ["B", "D"]
    grafo.remover_aresta("A", "B")
    assert grafo.existe_aresta("A", "B") is False
    assert grafo.existe_aresta("C", "B") is True
    assert grafo.peso("C", "B") == 0
    assert grafo.grau_entrada("B") == 1

lista_rotas = montar(GrafoLista, vertices, rotas, direcionado=True, ponderado=True)
matriz_rotas = montar(GrafoMatriz, vertices, rotas, direcionado=True, ponderado=True)
assert lista_rotas.espaco_estrutural() == 10
assert matriz_rotas.como_tabela() == [
    [None, 3, 1, None],
    [None, None, None, 4],
    [None, 0, None, 2.5],
    [None, None, None, None],
]
assert conteudo(lista_rotas.para_matriz()) == conteudo(matriz_rotas)
assert conteudo(matriz_rotas.para_lista()) == conteudo(lista_rotas)
assert matriz_rotas.para_lista().para_matriz().como_tabela() == matriz_rotas.como_tabela()

# Não direcionado com pesos: os dois sentidos carregam o mesmo número.
triangulo = montar(
    GrafoLista,
    ["X", "Y", "Z"],
    [("X", "Y", 10), ("Y", "Z", 4), ("Z", "X", 7)],
    ponderado=True,
)
triangulo_m = triangulo.para_matriz()
assert triangulo.peso("Y", "X") == 10
assert triangulo_m.peso("Z", "Y") == 4
assert triangulo_m.como_tabela() == [
    [None, 10, 7],
    [10, None, 4],
    [7, 4, None],
]
assert conteudo(triangulo) == conteudo(triangulo_m)

# Erros de uso.
vazio = GrafoLista()
assert vazio.vertices() == []
assert vazio.quantidade_arestas() == 0
assert "(grafo vazio)" in str(vazio)
exige(KeyError, lambda: vazio.adicionar_aresta("A", "B"))
exige(KeyError, lambda: vazio.remover_vertice("A"))
exige(KeyError, lambda: vazio.grau("A"))

base = montar(GrafoMatriz, vertices, arestas)
base.adicionar_vertice("A")
assert base.vertices() == vertices
exige(ValueError, lambda: base.adicionar_aresta("A", "A"))
exige(ValueError, lambda: base.adicionar_aresta("A", "B"))
exige(ValueError, lambda: base.adicionar_aresta("A", "D", 5))
exige(KeyError, lambda: base.remover_aresta("A", "D"))
exige(KeyError, lambda: base.peso("A", "D"))
exige(KeyError, lambda: base.remover_vertice("Z"))
exige(TypeError, lambda: base.adicionar_aresta("A", "D", True))

ponderado = GrafoLista(direcionado=True, ponderado=True)
ponderado.adicionar_vertice("A")
ponderado.adicionar_vertice("B")
exige(ValueError, lambda: ponderado.adicionar_aresta("A", "A", 3))
ponderado.adicionar_aresta("A", "B", 0)
exige(ValueError, lambda: ponderado.adicionar_aresta("A", "B", 1))

import io
from contextlib import redirect_stdout

buffer = io.StringIO()
with redirect_stdout(buffer):
    lista_aula.imprimir()
    matriz_aula.imprimir()
texto = buffer.getvalue()
assert "Lista de adjacência | não direcionado | sem pesos" in texto
assert "A → B, C" in texto
assert "Matriz de adjacência | não direcionado | sem pesos" in texto
assert "0" in texto and "1" in texto

print("Todos os testes passaram!")


Todos os testes passaram!


## Onde isso aparece

Os dois exemplos usam as mesmas classes. Os números de distância e de tempo são ilustrativos: servem para mostrar a estrutura, não como medição de trânsito.

**Rede social.** Amizade é um grafo não direcionado e sem pesos: a aresta não tem dono nem custo. O grau é o número de amigos. Seguir alguém é direcionado: o grau de saída é quantas pessoas a conta segue, e o grau de entrada é quantos seguidores ela tem. Uma rede grande é esparsa (cada pessoa conhece poucas, perto do total de usuários), então a lista de adjacência evita a tabela `V²`.

**Mapas.** Estrada de mão dupla é um grafo não direcionado com peso: o peso pode ser a distância em quilômetros. Rua de mão única é direcionado e com peso: o peso pode ser o tempo em minutos, e a volta pode nem existir. Num cruzamento pequeno, em que o programa consulta o tempo todo se duas esquinas se ligam, a matriz responde em O(1). Num mapa de cidade, com muitas esquinas e poucas ruas saindo de cada uma, a lista gasta menos memória.


In [9]:
# Amizade: não direcionado, sem pesos. A lista combina com uma rede esparsa.
amizades = GrafoLista(direcionado=False, ponderado=False)
for pessoa in ["Ana", "Bruno", "Carla", "Diego"]:
    amizades.adicionar_vertice(pessoa)
amizades.adicionar_aresta("Ana", "Bruno")
amizades.adicionar_aresta("Ana", "Carla")
amizades.adicionar_aresta("Bruno", "Diego")

print("Amizades")
amizades.imprimir()
print(f"Ana tem {amizades.grau('Ana')} amigos: {amizades.vizinhos('Ana')}")
assert amizades.grau("Ana") == 2
assert amizades.existe_aresta("Bruno", "Ana") is True
assert amizades.existe_aresta("Carla", "Diego") is False

# Seguir: direcionado, sem pesos. Entrada = seguidores; saída = seguindo.
seguir = GrafoLista(direcionado=True, ponderado=False)
for pessoa in ["Ana", "Bruno", "Carla"]:
    seguir.adicionar_vertice(pessoa)
seguir.adicionar_aresta("Bruno", "Ana")
seguir.adicionar_aresta("Carla", "Ana")
seguir.adicionar_aresta("Ana", "Carla")

print()
print("Quem segue quem")
seguir.imprimir()
print(f"Seguidores de Ana: {seguir.predecessores('Ana')}")
print(
    f"Ana segue {seguir.grau_saida('Ana')} e tem "
    f"{seguir.grau_entrada('Ana')} seguidores."
)
assert seguir.grau_entrada("Ana") == 2
assert seguir.grau_saida("Ana") == 1
assert seguir.existe_aresta("Ana", "Bruno") is False
assert seguir.existe_aresta("Bruno", "Ana") is True

# Mapa de mão dupla, peso = quilômetros ilustrativos.
distancias = GrafoLista(direcionado=False, ponderado=True)
for cidade in ["Natal", "Parnamirim", "Macaíba", "São Gonçalo"]:
    distancias.adicionar_vertice(cidade)
distancias.adicionar_aresta("Natal", "Parnamirim", 18)
distancias.adicionar_aresta("Natal", "Macaíba", 25)
distancias.adicionar_aresta("Parnamirim", "São Gonçalo", 40)
distancias.adicionar_aresta("Macaíba", "São Gonçalo", 30)

print()
print("Distâncias ilustrativas (km)")
distancias.imprimir()
print(f"Natal — Parnamirim: {distancias.peso('Natal', 'Parnamirim')} km")
assert distancias.peso("Parnamirim", "Natal") == 18
assert distancias.grau("Natal") == 2
assert distancias.existe_aresta("Natal", "São Gonçalo") is False

# Ruas de mão única entre quatro pontos, peso = minutos ilustrativos.
# A matriz deixa a consulta de cada par em O(1), útil num trecho pequeno.
minutos = GrafoMatriz(direcionado=True, ponderado=True)
for ponto in ["P1", "P2", "P3", "P4"]:
    minutos.adicionar_vertice(ponto)
minutos.adicionar_aresta("P1", "P2", 4)
minutos.adicionar_aresta("P2", "P4", 6)
minutos.adicionar_aresta("P1", "P3", 3)
minutos.adicionar_aresta("P3", "P4", 5)

print()
print("Ruas de mão única (min)")
minutos.imprimir()
assert minutos.existe_aresta("P1", "P2") is True
assert minutos.existe_aresta("P2", "P1") is False
assert minutos.peso("P3", "P4") == 5
assert minutos.grau_entrada("P4") == 2
assert minutos.para_lista().peso("P1", "P3") == 3

print()
print("Rede esparsa pede lista; trecho pequeno e muito consultado pode usar matriz.")


Amizades
Lista de adjacência | não direcionado | sem pesos
Ana → Bruno, Carla
Bruno → Ana, Diego
Carla → Ana
Diego → Bruno
Ana tem 2 amigos: ['Bruno', 'Carla']

Quem segue quem
Lista de adjacência | direcionado | sem pesos
Ana → Carla
Bruno → Ana
Carla → Ana
Seguidores de Ana: ['Bruno', 'Carla']
Ana segue 1 e tem 2 seguidores.

Distâncias ilustrativas (km)
Lista de adjacência | não direcionado | com pesos
Natal → Parnamirim(18), Macaíba(25)
Parnamirim → Natal(18), São Gonçalo(40)
Macaíba → Natal(25), São Gonçalo(30)
São Gonçalo → Parnamirim(40), Macaíba(30)
Natal — Parnamirim: 18 km

Ruas de mão única (min)
Matriz de adjacência | direcionado | com pesos
    P1  P2  P3  P4
P1   -   4   3   -
P2   -   -   -   6
P3   -   -   -   5
P4   -   -   -   -

Rede esparsa pede lista; trecho pequeno e muito consultado pode usar matriz.


## Resumo

- Um grafo é vértices mais arestas. A aresta pode ter sentido e pode ter peso.
- A **lista de adjacência** guarda, para cada vértice, só quem está ligado a ele. Espaço **O(V + E)**. Perguntar se uma aresta existe custa **O(grau)**. É a representação econômica para grafos esparsos, como redes sociais e mapas.
- A **matriz de adjacência** guarda uma casa para cada par. Espaço **O(V²)**, consulta da aresta em **O(1)**. No exemplo da aula a tabela é simétrica e a casa (A, D) fica 0.
- As duas classes deste notebook fazem as mesmas operações. `para_lista` e `para_matriz` copiam um grafo de uma representação para a outra sem mudar sentido nem peso.
- O peso 0 é aresta de custo nulo. A casa vazia, no grafo ponderado, não é 0.

O grafo da aula, para conferir de um relance:

```text
A → B, C
B → A, C, D
C → A, B, D
D → B, C

    A  B  C  D
A   0  1  1  0
B   1  0  1  1
C   1  1  0  1
D   0  1  1  0
```
